In [ ]:
!wget -q -O CEAS_08.csv "https://zenodo.org/records/8339691/files/CEAS_08.csv?download=1"
!wget -q -O Nazario.csv "https://zenodo.org/records/8339691/files/Nazario.csv?download=1"
!wget -q -O SpamAssasin.csv "https://zenodo.org/records/8339691/files/SpamAssasin.csv?download=1"
!ls -lh *.csv

-rw-r--r-- 1 root root  65M Oct  7 17:38 CEAS_08.csv
-rw-r--r-- 1 root root 7.5M Oct  7 17:38 Nazario.csv
-rw-r--r-- 1 root root  15M Oct  7 17:38 SpamAssasin.csv


In [ ]:
import pandas as pd, numpy as np, re, random

files = {"CEAS_08.csv": "ceas", "Nazario.csv": "nazario", "SpamAssasin.csv": "spamassassin"}
need = ["sender", "subject", "body", "label"]

frames = []
for f, name in files.items():
    d = pd.read_csv(f)
    missing = [c for c in need if c not in d.columns]
    assert not missing, f"{f} is missing columns {missing}. Columns are {list(d.columns)}"
    d = d[need].copy()
    d["source"] = name
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
raw = raw.dropna(subset=["body", "label"])
raw["label"] = raw["label"].astype(int)
print(raw.shape)
print(raw.groupby("source")["label"].value_counts())

(46527, 5)
source        label
ceas          1        21842
              0        17312
nazario       1         1565
spamassassin  0         4091
              1         1717
Name: count, dtype: int64


In [ ]:
def get_domain(s):
    m = re.search(r"@([\w\.-]+)", str(s))
    return m.group(1).lower() if m else "unknown"

def clean_body(t):
    t = re.sub(r"<[^>]+>", " ", str(t))                # remove HTML tags
    t = re.sub(r"http\S+|www\.\S+", " [URL] ", t)      # links -> [URL]
    t = re.sub(r"\S+@\S+", " [EMAIL] ", t)             # email addresses -> [EMAIL]
    return re.sub(r"\s+", " ", t).strip()

raw["body_raw"] = raw["body"]                           # keep original for the brand/link check later
raw["body"] = raw["body"].apply(clean_body)
raw["subject"] = raw["subject"].fillna("").astype(str)
raw["domain"] = raw["sender"].apply(get_domain)

raw = raw[raw["body"].str.split().str.len() >= 5]       # drop empty emails

raw["text"] = ("from: " + raw["domain"] + " | subject: " + raw["subject"]
               + " | " + raw["body"].str[:1500])

n0 = len(raw)
raw = raw.drop_duplicates(subset="text")
print("duplicates removed:", n0 - len(raw))

nl = raw.groupby("text")["label"].nunique()
bad = nl[nl > 1].index
raw = raw[~raw["text"].isin(bad)]
print("conflicting-label texts removed:", len(bad))
print("final:", raw.shape)
print(raw["label"].value_counts())

duplicates removed: 689
conflicting-label texts removed: 0
final: (45042, 8)
label
1    23853
0    21189
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(raw, test_size=0.2, stratify=raw["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)
print(len(train_df), len(val_df), len(test_df))

36033 4504 4505


In [16]:
random.seed(7)

brands = {
    "Netflix": ["netflix.com"], "Amazon": ["amazon.com", "amazon.in"],
    "Google Workspace": ["google.com"], "PayPal": ["paypal.com"],
    "HDFC Bank": ["hdfcbank.com"], "SBI": ["sbi.co.in"], "ICICI Bank": ["icicibank.com"],
    "Microsoft": ["microsoft.com"], "Apple": ["apple.com"], "Flipkart": ["flipkart.com"],
    "Airtel": ["airtel.in"], "Paytm": ["paytm.com"], "Swiggy": ["swiggy.com"],
}
free = ["gmail.com", "outlook.com", "yahoo.com", "hotmail.com"]
generic_fake = ["secure-alerts.xyz", "verify-now.top", "prize-claims.info", "customs-fee.online", "support-desk.co", "refund-center.top"]

scam_train = [
 "Your {b} account will be suspended in {h} hours. Verify your details immediately to avoid permanent closure. [URL]",
 "Immediate payment of Rs {a} required. Failure to pay will result in a penalty notice and legal action. Pay now: [URL]",
 "KYC update required. Your bank account is locked until you complete verification within {h} hours. [URL]",
 "Final warning: your {b} payment failed. Update your billing information now or your account will be terminated in {h} hours. [URL]",
 "Congratulations, you are selected for a work from home job paying Rs {a} per day. Pay a registration fee of Rs 1500 to confirm your slot today. [URL]",
 "You have won a prize of Rs {a}. To claim it, send your bank details and a small processing fee within {h} hours or the prize will be cancelled.",
 "Your parcel could not be delivered. Pay a customs fee of Rs 49 using this link within {h} hours or it will be returned to the sender. [URL]",
 "We have a refund of Rs {a} pending for you. Share your card number and the OTP you received to get it now. [URL]",
 "Your {b} debit card has been blocked. Call the number below or click the link to reactivate it today or your account will be frozen. [URL]",
 "Urgent: your {b} password expires in {h} hours. Click here to keep your access or you will lose all your data. [URL]",
 "Dear customer, your electricity will be disconnected tonight at 9:30 pm because your last bill was not updated. Contact the officer immediately on the number below.",
 "Income tax department: a pending refund of Rs {a} is on hold. Confirm your PAN and bank login within {h} hours to avoid a penalty. [URL]",
 "Your {b} UPI payment request of Rs {a} is pending. Approve it now to receive your cashback. [URL]",
]
scam_test = [
 "Your SIM will be blocked in {h} hours due to incomplete KYC. Send a photo of your ID and the OTP to continue the service.",
 "Limited slots: invest Rs {a} today and receive double returns in {h} hours. Guaranteed profit, act now before the offer closes. [URL]",
 "Security alert from {b}: someone tried to log in to your account. Confirm your identity within {h} hours using the link below or your account will be deactivated. [URL]",
 "Your {b} order of Rs {a} will be charged to your card today. If you did not place this order, call our helpline immediately to cancel.",
 "Final notice: your loan EMI of Rs {a} is overdue. Pay today to avoid a legal notice and a visit to your home. [URL]",
 "Your {b} subscription has expired. Renew now with a one time payment of Rs {a} by gift card to restore access. [URL]",
]
safe_train = [
 "Hi, your {b} order has been shipped and will arrive on Friday. You can track your package in your account. Thanks for shopping with us.",
 "Your monthly {b} statement is now available. No action is needed. You can view it anytime by logging in to the official app.",
 "Reminder: your {b} subscription renews on the 15th for Rs {a}. You can manage or cancel it anytime in your account settings.",
 "Thanks for your payment of Rs {a}. Your receipt is attached. If you have questions, reply to this email or visit our help center.",
 "We have updated our privacy policy. You do not need to do anything. You can read the changes on our website whenever you like.",
 "Your KYC was successfully verified. No further action is required. Thank you for completing the process with {b}.",
 "Security tip from {b}: we never ask for your password by email. If you see a suspicious message, report it from your account.",
 "Your OTP for the transaction of Rs {a} is 482913. It is valid for 10 minutes. Do not share this OTP with anyone. {b} never asks for your OTP.",
 "Your {b} bill of Rs {a} is due on the 15th. Pay by the due date to avoid a late fee. You can pay in the official app.",
 "Hurry, the sale ends tonight! Get up to 50% off on selected items at {b}. Shop now and enjoy free delivery.",
 "We noticed a new sign-in to your {b} account on a Windows device. If this was you, you do not need to do anything.",
 "Hi, I wanted to confirm our interview tomorrow at 10 am. Please reply to confirm or let me know a better time.",
 "Hi team, just a reminder that the project review is on Monday at 11 am. Please bring the latest report and send me any questions.",
 "Your {b} password was changed successfully. If you made this change, no action is needed. If not, contact support from the official website.",
]
safe_test = [
 "Your package from {b} was delivered today at 3 pm. If anything is missing, you can contact support within 7 days.",
 "Your one time password for logging in to {b} is 739102. It expires in 5 minutes. Never share it with anyone.",
 "Weekly digest: here are the top stories picked for you this week. You can unsubscribe at any time using the link below.",
 "Payment reminder: your {b} invoice for Rs {a} is due next week. Please pay by bank transfer at your convenience. Thank you.",
 "Flash sale: 24 hours only! Enjoy 30% off at {b}. Offer ends tomorrow at midnight.",
 "Hi, can you send me the notes from yesterday's class when you get a chance? Thanks!",
]
scam_subj = ["Urgent: action required", "Account suspended", "Final notice", "Penalty notice", "KYC update required", "You have won", "Refund pending", "Package on hold", "Card blocked", "Job offer"]
safe_subj = ["Your order update", "Your monthly statement", "Payment receipt", "Subscription reminder", "Privacy policy update", "OTP", "Bill due", "Sale ends tonight", "Security alert", "Meeting tomorrow"]

def make(templates, n, label):
    rows = []
    for _ in range(n):
        t = random.choice(templates)
        b = random.choice(list(brands))
        body = t.format(b=b, h=random.choice([2, 12, 24, 48]), a=random.choice([199, 499, 999, 1999, 4500, 9800]))
        uses_brand = "{b}" in t
        if label == 1:
            if random.random() < 0.3:
                dom = random.choice(free)
            elif uses_brand:
                dom = b.lower().replace(" ", "") + "-" + random.choice(["billing", "secure", "support", "verify", "alerts"]) + "." + random.choice(["xyz", "top", "info", "online", "co"])
            else:
                dom = random.choice(generic_fake)
            subj = random.choice(scam_subj)
        else:
            dom = random.choice(brands[b]) if uses_brand else random.choice(free + ["college.edu.in", "company.com"])
            subj = random.choice(safe_subj)
        rows.append({"text": f"from: {dom} | subject: {subj} | {body}", "label": label})
    return pd.DataFrame(rows).drop_duplicates(subset="text")

syn_train = pd.concat([make(scam_train, 1500, 1), make(safe_train, 1500, 0)], ignore_index=True)
syn_test  = pd.concat([make(scam_test, 300, 1),  make(safe_test, 300, 0)],  ignore_index=True)
syn_test = syn_test[~syn_test["text"].isin(syn_train["text"])]

train_df2 = pd.concat([train_df[["text", "label"]], syn_train], ignore_index=True).sample(frac=1, random_state=42)
print("synthetic train:", len(syn_train), "| held-out synthetic test:", len(syn_test), "| train size now:", len(train_df2))

synthetic train: 2441 | held-out synthetic test: 537 | train size now: 38474


In [17]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
Xtr = vec.fit_transform(train_df2["text"])
base = LogisticRegression(max_iter=1000, C=4)
base.fit(Xtr, train_df2["label"])

print(classification_report(val_df["label"], base.predict(vec.transform(val_df["text"])), digits=4))

              precision    recall  f1-score   support

           0     0.9883    0.9934    0.9908      2119
           1     0.9941    0.9895    0.9918      2385

    accuracy                         0.9913      4504
   macro avg     0.9912    0.9915    0.9913      4504
weighted avg     0.9914    0.9913    0.9913      4504



In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

MODEL = "distilbert-base-uncased"
MAXLEN = 128
tok = AutoTokenizer.from_pretrained(MODEL)

class EmailDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAXLEN)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

train_ds = EmailDS(train_df2["text"], train_df2["label"])
val_ds   = EmailDS(val_df["text"],    val_df["label"])
test_ds  = EmailDS(test_df["text"],   test_df["label"])
print("ready:", len(train_ds), len(val_ds), len(test_ds))

ready: 38474 4504 4505


In [19]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary")
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="out",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

# save to Google Drive so a timeout can't delete the model
from google.colab import drive
drive.mount('/content/drive')
trainer.save_model("/content/drive/MyDrive/scam_model")
tok.save_pretrained("/content/drive/MyDrive/scam_model")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.033290,0.025120,0.992673,0.996622,0.989518,0.993057
2,0.005402,0.020424,0.994893,0.996218,0.994130,0.995173


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/scam_model/tokenizer_config.json',
 '/content/drive/MyDrive/scam_model/tokenizer.json')

In [20]:
from scipy.special import softmax

val_logits = trainer.predict(val_ds).predictions
val_prob = softmax(val_logits, axis=1)[:, 1]

best_t, best_f1 = 0.5, 0
for t in np.arange(0.1, 0.91, 0.05):
    f = precision_recall_fscore_support(val_df["label"], (val_prob >= t).astype(int), average="binary")[2]
    if f > best_f1:
        best_t, best_f1 = t, f
print("best threshold:", round(best_t, 2), "val F1:", round(best_f1, 4))

test_logits = trainer.predict(test_ds).predictions
test_prob = softmax(test_logits, axis=1)[:, 1]
test_pred = (test_prob >= best_t).astype(int)

print("\nDistilBERT on TEST:")
print(classification_report(test_df["label"], test_pred, digits=4))
print("Baseline on TEST:")
print(classification_report(test_df["label"], base.predict(vec.transform(test_df["text"])), digits=4))

best threshold: 0.3 val F1: 0.996



DistilBERT on TEST:
              precision    recall  f1-score   support

           0     0.9967    0.9972    0.9969      2119
           1     0.9975    0.9971    0.9973      2386

    accuracy                         0.9971      4505
   macro avg     0.9971    0.9971    0.9971      4505
weighted avg     0.9971    0.9971    0.9971      4505

Baseline on TEST:
              precision    recall  f1-score   support

           0     0.9934    0.9920    0.9927      2119
           1     0.9929    0.9941    0.9935      2386

    accuracy                         0.9931      4505
   macro avg     0.9931    0.9931    0.9931      4505
weighted avg     0.9931    0.9931    0.9931      4505



In [ ]:
errs = test_df.copy()
errs["pred"] = test_pred
errs["prob"] = test_prob
errs = errs[errs["pred"] != errs["label"]]
print("wrong:", len(errs), "of", len(test_df))
for _, r in errs.sample(min(15, len(errs)), random_state=1).iterrows():
    print(f"true={r['label']} pred={r['pred']} p={r['prob']:.2f} | {r['source']} | {r['text'][:200]}\n")

trainer.save_model("scam_model")
tok.save_pretrained("scam_model")
!zip -qr scam_model.zip scam_model

wrong: 22 of 4505
true=0 pred=1 p=0.84 | spamassassin | from: yesmail.com | subject: RE: The Curse of India's Socialism | This message is in MIME format. Since your mail reader does not understand this format, some or all of this message may not be legible

true=1 pred=0 p=0.00 | ceas | from: server7.businessworldlist.com | subject: rent, Baseball and Marketing. | Thu, August 7, 2008 Hi rent, Yesterday, I learned yet another life's little lesson while playing catch with my son, Adam 

true=0 pred=1 p=0.90 | spamassassin | from: matrox.com | subject: =?iso-8859-1?Q?Matrox_Parhelia=99_now_available?= | --=====================_22773175==_.ALT Content-Type: text/plain; charset="us-ascii"; format=flowed Matrox Parhelia - av

true=1 pred=0 p=0.32 | spamassassin | from: yahoo.com | subject: Hi,  Low price  inkjet cartridges bvax | HI, [EMAIL] today, Ink Price ________________________________________________________________________________________ If you would n

true=0 pred=1 p=0.94 | ceas | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
fn = np.array(vec.get_feature_names_out())
coef = base.coef_[0]
print("TOP features -> SCAM (1):\n", list(fn[np.argsort(coef)[-30:]][::-1]))
print("\nTOP features -> SAFE (0):\n", list(fn[np.argsort(coef)[:30]]))

TOP features -> SCAM (1):
 ['your', 'love', 'subject', 'watches', 'replica', 'you', 'our', 'from', 'com subject', 'life', 'men', 'her', 'account', 'monkey org', 'monkey', 'com', 'health', 'subject from', 'click', 'money', '000', 'will', 'we', 'love is', 'you will', 'replica watches', 'increase', 'for you', 'prince', 'guaranteed']

TOP features -> SAFE (0):
 ['wrote', 'the', 're', 'subject re', 'thanks', '2008', 'gmail com', 'from gmail', 'gmail', 'but', '2007', 'tony', 'on', 'python', 'opensuse', 'bush', 'or', 'spamassassin', 'perl', 'uai', 'february', '08', 'nz', 'that', 'subscription', 'nz subject', '11', 'this', 'code', 'newsletter']


In [ ]:
t = test_df.copy()
t["pred"] = test_pred
for s, g in t.groupby("source"):
    print(s, "| rows:", len(g), "| accuracy:", round((g.pred == g.label).mean(), 4), "| labels:", g.label.value_counts().to_dict())

ceas | rows: 3722 | accuracy: 0.9973 | labels: {1: 2045, 0: 1677}
nazario | rows: 159 | accuracy: 0.9937 | labels: {1: 159}
spamassassin | rows: 624 | accuracy: 0.9824 | labels: {0: 442, 1: 182}


In [ ]:
strip = lambda s: s.str.replace(r"^from: .*? \| ", "", regex=True)

vec2 = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
b2 = LogisticRegression(max_iter=1000, C=4).fit(vec2.fit_transform(strip(train_df2["text"])), train_df2["label"])
print("WITHOUT sender:")
print(classification_report(test_df["label"], b2.predict(vec2.transform(strip(test_df["text"]))), digits=4))

# using ONLY the sender domain
dom = lambda s: s.str.extract(r"^from: (.*?) \|")[0].fillna("unknown")
vec3 = TfidfVectorizer()
b3 = LogisticRegression(max_iter=1000).fit(vec3.fit_transform(dom(train_df2["text"])), train_df2["label"])
print("ONLY sender domain:")
print(classification_report(test_df["label"], b3.predict(vec3.transform(dom(test_df["text"]))), digits=4))

WITHOUT sender:
              precision    recall  f1-score   support

           0     0.9920    0.9906    0.9913      2119
           1     0.9916    0.9929    0.9923      2386

    accuracy                         0.9918      4505
   macro avg     0.9918    0.9917    0.9918      4505
weighted avg     0.9918    0.9918    0.9918      4505

ONLY sender domain:
              precision    recall  f1-score   support

           0     0.6405    0.8797    0.7413      2119
           1     0.8401    0.5616    0.6732      2386

    accuracy                         0.7112      4505
   macro avg     0.7403    0.7206    0.7072      4505
weighted avg     0.7463    0.7112    0.7052      4505



In [ ]:
m = trainer.model.eval()

def predict(texts):
    enc = tok(texts, truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt").to(m.device)
    with torch.no_grad():
        return torch.softmax(m(**enc).logits, dim=1)[:, 1].cpu().numpy()

tests = [
 ("from: amazon.in | subject: Your order has shipped | Hi Shruti, your order #4021 has shipped and will arrive on Friday. Track it in your Amazon account. Thanks for shopping with us.", 0),
 ("from: gmail.com | subject: Meeting tomorrow | Hi team, just a reminder that the project review is tomorrow at 11 am. Please bring the latest report and let me know if the time does not work.", 0),
 ("from: hdfcbank.com | subject: Your monthly statement | Your account statement for September is now available in net banking. No action is needed from your side.", 0),
 ("from: sbi-secure-kyc.top | subject: KYC update required | Dear customer, your bank account is locked. Complete KYC update within 24 hours or your account will be permanently suspended. Click the link to verify now.", 1),
 ("from: netflix-billing.xyz | subject: Payment failed | Your Netflix account will be suspended in 24 hours. Update your payment details immediately to avoid cancellation.", 1),
 ("from: gmail.com | subject: Immediate payment required | Penalty notice: you owe Rs 4500. Pay immediately by transferring to the account below to avoid legal action today.", 1),
 ("from: paypal.com | subject: Receipt for your payment | Thanks for your payment of Rs 499. Your receipt is attached. Reply to this email if you have any questions.", 0),
 ("from: outlook.com | subject: Congratulations winner | You have won a lottery of Rs 50,00,000. Send your bank details and a processing fee now to claim the prize before it expires.", 1),
]
probs = predict([x[0] for x in tests])
for (txt, true), p in zip(tests, probs):
    print(f"true={true} | scam_prob={p:.2f} | {'OK' if (p >= best_t) == bool(true) else 'WRONG'} | {txt[:70]}...")

true=0 | scam_prob=0.00 | OK | from: amazon.in | subject: Your order has shipped | Hi Shruti, your or...
true=0 | scam_prob=0.00 | OK | from: gmail.com | subject: Meeting tomorrow | Hi team, just a reminder...
true=0 | scam_prob=0.28 | OK | from: hdfcbank.com | subject: Your monthly statement | Your account st...
true=1 | scam_prob=1.00 | OK | from: sbi-secure-kyc.top | subject: KYC update required | Dear custome...
true=1 | scam_prob=1.00 | OK | from: netflix-billing.xyz | subject: Payment failed | Your Netflix acc...
true=1 | scam_prob=0.95 | OK | from: gmail.com | subject: Immediate payment required | Penalty notice...
true=0 | scam_prob=0.00 | OK | from: paypal.com | subject: Receipt for your payment | Thanks for your...
true=1 | scam_prob=1.00 | OK | from: outlook.com | subject: Congratulations winner | You have won a l...


In [22]:
hard = [
 ("from: icicibank.com | subject: OTP for your transaction | Your OTP for the transaction of Rs 2,499 is 483920. It is valid for 10 minutes. Do not share this OTP with anyone. The bank never asks for your OTP.", 0),
 ("from: airtel.in | subject: Bill payment due | Your Airtel bill of Rs 799 is due on 15 October. Pay by the due date to avoid a late fee. You can pay in the Airtel Thanks app.", 0),
 ("from: amazon.in | subject: Limited time offer: 50% off | Hurry, the sale ends tonight. Get up to 50% off on electronics. Shop now on Amazon.", 0),
 ("from: google.com | subject: Security alert | We noticed a new sign-in to your Google Account on a Windows device. If this was you, you do not need to do anything.", 0),
 ("from: gmail.com | subject: Interview tomorrow | Hi Shruti, we would like to confirm your interview tomorrow at 10 am. Please reply to confirm or tell me a better time.", 0),
 ("from: gmail.com | subject: Job offer | Congratulations, you are selected for a work from home job paying Rs 5000 per day. Pay a registration fee of Rs 1500 to confirm your slot today.", 1),
 ("from: sbi-alerts.in | subject: Card blocked | Your debit card has been blocked. Click the link to reactivate within 2 hours or your account will be frozen.", 1),
 ("from: indiapost-delivery.top | subject: Package on hold | Your parcel could not be delivered. Pay a customs fee of Rs 49 using this link to release it, otherwise it will be returned.", 1),
 ("from: amazon-support.co | subject: Refund pending | We have a refund of Rs 3200 pending for you. Share your card number and the OTP you received to receive it now.", 1),
]
probs = predict([x[0] for x in hard])
print("threshold used:", round(best_t, 2))
for (txt, true), p in zip(hard, probs):
    print(f"true={true} | scam_prob={p:.2f} | {'OK' if (p >= best_t) == bool(true) else 'WRONG'} | {txt[:75]}...")

threshold used: 0.3
true=0 | scam_prob=0.00 | OK | from: icicibank.com | subject: OTP for your transaction | Your OTP for the ...
true=0 | scam_prob=0.00 | OK | from: airtel.in | subject: Bill payment due | Your Airtel bill of Rs 799 is...
true=0 | scam_prob=0.00 | OK | from: amazon.in | subject: Limited time offer: 50% off | Hurry, the sale en...
true=0 | scam_prob=0.00 | OK | from: google.com | subject: Security alert | We noticed a new sign-in to yo...
true=0 | scam_prob=0.00 | OK | from: gmail.com | subject: Interview tomorrow | Hi Shruti, we would like to...
true=1 | scam_prob=1.00 | OK | from: gmail.com | subject: Job offer | Congratulations, you are selected fo...
true=1 | scam_prob=1.00 | OK | from: sbi-alerts.in | subject: Card blocked | Your debit card has been bloc...
true=1 | scam_prob=1.00 | OK | from: indiapost-delivery.top | subject: Package on hold | Your parcel could...
true=1 | scam_prob=1.00 | OK | from: amazon-support.co | subject: Refund pending | We have a refund 

In [21]:
m = trainer.model.eval()          # use the NEW trained model

syn_test_ds = EmailDS(syn_test["text"], syn_test["label"])
p = softmax(trainer.predict(syn_test_ds).predictions, axis=1)[:, 1]
print("Held-out synthetic test (wordings never seen in training):")
print(classification_report(syn_test["label"], (p >= best_t).astype(int), digits=4))

Held-out synthetic test (wordings never seen in training):
              precision    recall  f1-score   support

           0     0.9101    0.9918    0.9492       245
           1     0.9926    0.9178    0.9537       292

    accuracy                         0.9516       537
   macro avg     0.9514    0.9548    0.9515       537
weighted avg     0.9550    0.9516    0.9517       537

